# Лабораторная работа №1. Анализ текстов (NLP / Text Mining). Васильева Светлана


## Задание (по методическим указаниям)
1. Определить задачу анализа текстов.  
2. Применить методы предобработки текста и сравнить варианты.  
3. Оценить влияние POS tagging и других морфологических признаков.  
4. Применить методы нормализации и сравнить варианты.  
5. Применить методы фильтрации и сравнить варианты.  
6. Оценить влияние формирования N‑грамм.  
7. Выполнить индексирование текста (BoW, TF‑IDF) и сравнить подходы.  
8. Обучить модель с учителем и оценить качество для различных комбинаций гиперпараметров.  
9. Оформить код, промежуточные результаты и выводы в виде Jupyter Notebook.


## Инструменты
`spaCy` (токенизация, лемматизация, POS), `scikit-learn` (векторизация, модели, кросс‑валидация), при необходимости `NLTK` (стемминг).

## Пункт 1. Постановка задачи (с учителем)

### Формулировка задачи
Требуется решить задачу **многоклассовой классификации** текстовых фрагментов (абзацев) по разделам технического задания.

### Данные
- **Источник:** файл `tz.docx` (русскоязычный документ ТЗ).  
- **Формирование примеров:** документ разбивается на абзацы; каждому абзацу присваивается метка `label` по текущему заголовку раздела.  
- **Признак:** `text` — текст абзаца.  
- **Целевая переменная:** `label` — идентификатор раздела (например, `2.2.1`, `2.2.3.2` и т.п.).

### Метрики качества
Используются `Accuracy` и `macro‑F1`. При наличии дисбаланса классов основной показатель — `macro‑F1` (равный вклад классов).


## Пункт 2. Предобработка текста и сравнение конфигураций

На данном этапе формируется корпус из файла `tz.docx` и выполняется сравнение нескольких конфигураций предобработки (трансформация исходного текста, токенизация, базовые правила очистки). 

 Качество оценивается кросс‑валидацией (StratifiedKFold) по метрикам `Accuracy` и `macro‑F1`.

In [45]:
import re
import pandas as pd
import spacy
from docx import Document

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC


nlp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])

CHAPTER_MAP = {
    "2.2.1": "Общие сведения",
    "2.2.2": "Назначение и цели",
    "2.2.3.1": "Требования к системе",
    "2.2.3.2": "Требования к функциям",
    "2.2.3.3": "Требования к обеспечению",
    "2.2.4": "Этапы работ",
    "2.2.5": "Исполнители работ",
    "2.2.6": "Подготовка к вводу",
    "2.2.7": "Документирование",
}
ALLOWED = set(CHAPTER_MAP.keys())
SECTION_RE = re.compile(r"^\s*(\d+(?:\.\d+)*)(?:\.)?\s+.+$")

def load_dataset(docx_path: str) -> pd.DataFrame:
    doc = Document(docx_path)
    rows, current = [], None
    for p in doc.paragraphs:
        text = (p.text or "").strip()
        if not text:
            continue
        if SECTION_RE.match(text):
            sec = text.split()[0].rstrip(".")
            current = sec if sec in ALLOWED else None
            continue
        if current:
            rows.append({"text": text, "label": current})
    return pd.DataFrame(rows)

df = load_dataset("tz.docx")



### Подготовка выборки: исключение редких классов

Для корректной стратифицированной кросс‑валидации необходимо, чтобы в каждом классе было достаточно примеров. Поэтому классы с числом абзацев меньше заданного порога удаляются из выборки. Это уменьшает риск некорректного разбиения и нестабильных оценок качества.

In [46]:
MIN_SAMPLES_PER_CLASS = 3
counts = df["label"].value_counts()
keep_labels = counts[counts >= MIN_SAMPLES_PER_CLASS].index
df = df[df["label"].isin(keep_labels)].reset_index(drop=True)

print("После фильтра классов (>= 3 примеров):", df.shape)
print(df["label"].value_counts())



После фильтра классов (>= 3 примеров): (63, 2)
label
2.2.3.2    15
2.2.3.3    15
2.2.3.1    11
2.2.6      10
2.2.1       6
2.2.4       3
2.2.7       3
Name: count, dtype: int64


### Трансформация исходного текста
Базовой очистки текста (приведение регистра, обработка чисел, удаление/сохранение пунктуации, нормализация пробелов).

In [47]:
NUM_RE = re.compile(r"\d+")
PUNCT_RE = re.compile(r"[^0-9A-Za-zА-Яа-яЁё\s]+")
WS_RE = re.compile(r"\s+")
LISTNUM_RE = re.compile(r"^\s*\(?\d+(?:\.\d+)*\)?[).]?\s*")

def transform(text, lower=True, mask_numbers=False, remove_punct=False, remove_list_numbers=False):
    t = text
    if lower:
        t = t.lower()
    if remove_list_numbers:
        t = LISTNUM_RE.sub("", t)
    if mask_numbers:
        t = NUM_RE.sub(" NUM ", t)
    if remove_punct:
        t = PUNCT_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()



### Токенизация с использованием spaCy

Для векторизации требуется преобразовать текст в последовательность токенов. Используется `spaCy`, так как библиотека обеспечивает корректную токенизацию русского языка и предоставляет морфологические признаки, которые потребуются в последующих пунктах работы.

In [48]:
CONTENT_POS = {"NOUN", "ADJ", "VERB", "PROPN"}

def make_clean_text(text, mode):
    if mode == "all":
        text = transform(text, lower=True, remove_list_numbers=True, mask_numbers=True, remove_punct=True)
    else:
        text = transform(text, lower=True, remove_list_numbers=True)

    doc = nlp(text)
    tokens = []
    for tok in doc:
        if tok.is_space or tok.is_punct or tok.is_digit:
            continue
        if mode in ("nostop", "all") and tok.is_stop:
            continue
        if mode in ("pos", "all") and tok.pos_ not in CONTENT_POS:
            continue

        w = tok.lemma_ if mode in ("lemma", "all") else tok.text
        if len(w) >= 2:
            tokens.append(w)

    return " ".join(tokens)



### Набор конфигураций предобработки

Формируется несколько конфигураций, которые различаются интенсивностью очистки . Далее по таблице метрик выбирается конфигурация, обеспечивающая наилучший `macro‑F1`.

In [49]:
CONFIGS = [
    ("base",   dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("nostop", dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("lemma",  dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("pos",    dict(min_df=1, max_df=1.0, ngram_range=(1, 1))),
    ("all",    dict(min_df=2, max_df=0.9, ngram_range=(1, 2))),
]

X = df["text"].astype(str).tolist()
y = df["label"].astype(str).values



### Настройка кросс‑валидации

Число фолдов `n_splits` выбирается с учётом минимального размера класса (условие: `n_splits ≤ min_samples_per_class`). Это позволяет избежать предупреждений и ошибок разбиения при стратификации.

In [50]:
min_class = pd.Series(y).value_counts().min()
n_splits = 3 if min_class >= 3 else 2
if n_splits < 2:
    raise ValueError("Слишком мало данных: в каком-то классе меньше 2 примеров.")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print("CV folds:", n_splits, "| min_class:", min_class)

results = []
for mode, vec_params in CONFIGS:
    X_clean = [make_clean_text(t, mode) for t in X]

    pipe = make_pipeline(
        TfidfVectorizer(sublinear_tf=True, lowercase=False, **vec_params),
        LinearSVC(class_weight="balanced")
    )

    scores = cross_validate(
        pipe, X_clean, y, cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    results.append({
        "mode": mode,
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "vec": vec_params
    })

res = pd.DataFrame(results).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res)

CV folds: 3 | min_class: 3


,mode,accuracy,macro_f1,vec
2,lemma,0.619048,0.564793,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
4,all,0.650794,0.502220,"{'min_df': 2, 'max_df': 0.9, 'ngram_range': (1..."
1,nostop,0.603175,0.497023,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
0,base,0.603175,0.491760,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."
3,pos,0.587302,0.465701,"{'min_df': 1, 'max_df': 1.0, 'ngram_range': (1..."




### Вывод по пункту 2
Проведено сравнение нескольких вариантов предобработки при фиксированном конвейере обучения. По таблице `res_df` выбирается конфигурация с максимальным `macro‑F1`, которая далее используется как базовая для последующих экспериментов.

## Пункт 3. Влияние POS и морфологических признаков

Цель пункта — оценить, влияет ли использование морфологических признаков (в первую очередь частей речи) на качество классификации. Для этого фиксируются настройки предобработки и модели, а изменяется способ представления токенов:
- использование всех токенов;
- отбор токенов по POS (например, только содержательные части речи);
- включение POS в представление токена (пример: `lemma_POS`).

In [51]:
import pandas as pd
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC



In [52]:
nlp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])
print("spaCy pipeline:", nlp.pipe_names)



spaCy pipeline: ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']


### Фиксация базовых настроек эксперимента

Для корректного сравнения влияния POS фиксируются настройки предобработки (лучший вариант из пункта 2), модель и схема кросс‑валидации. В дальнейшем изменяется только способ формирования токенов средствами `spaCy`.

In [53]:
# ============================================
# Пункт 7–8: предобработка через transform(...)
# ============================================

BEST_PREPROCESS = dict(
    lower=True,
    mask_numbers=True,        # handle_numbers="mask" -> mask_numbers=True
    remove_punct=True,        # keep_punct=False -> remove_punct=True
    remove_list_numbers=False # remove_section_numbers=False
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values

Xp = pd.Series(X).apply(lambda s: transform(s, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
cv = StratifiedKFold(
    n_splits=5 if min_class >= 5 else 3,
    shuffle=True,
    random_state=42
)

### Преобразование spaCy‑токенов в текстовый ввод для векторизатора

Векторизаторы `scikit-learn` ожидают список токенов или строку. Поэтому ввод формируется на основе результатов `spaCy` (с учётом выбранных POS‑ограничений и/или добавления морфологических признаков).

In [54]:
def to_mode_text(text, allowed_pos=None, morph=None, repr_mode="text"):
    doc = nlp(text)

    tokens = [t for t in doc if not t.is_space and not t.is_punct and not t.is_digit]

    if allowed_pos:
        tokens = [t for t in tokens if t.pos_ in allowed_pos]

    if morph:
        for feat, allowed_vals in morph.items():
            tokens = [t for t in tokens if set(t.morph.get(feat)).intersection(allowed_vals)]

    if repr_mode == "lemma":
        words = [t.lemma_ for t in tokens]
    elif repr_mode == "lemma_pos":
        words = [f"{t.lemma_}_{t.pos_}" for t in tokens]
    else:
        words = [t.text for t in tokens]

    return " ".join([w for w in words if w])

CONTENT_POS = {"NOUN", "PROPN", "ADJ", "VERB", "ADV"}

MODES = [
    ("BASE_text_allPOS",      dict(allowed_pos=None,        morph=None,                repr_mode="text")),
    ("POS_content_text",      dict(allowed_pos=CONTENT_POS, morph=None,                repr_mode="text")),
    ("POS_noun_text",         dict(allowed_pos={"NOUN"},    morph=None,                repr_mode="text")),
    ("LEMMA_allPOS",          dict(allowed_pos=None,        morph=None,                repr_mode="lemma")),
    ("LEMMA_POS_allPOS",      dict(allowed_pos=None,        morph=None,                repr_mode="lemma_pos")),
    ("MORPH_NOUN_Case=Nom",   dict(allowed_pos={"NOUN"},    morph={"Case": {"Nom"}},   repr_mode="lemma")),
    ("MORPH_VERB_Tense=Past", dict(allowed_pos={"VERB"},    morph={"Tense": {"Past"}}, repr_mode="lemma")),
]

pipe = make_pipeline(
    TfidfVectorizer(sublinear_tf=True, lowercase=False),
    LinearSVC(class_weight="balanced")
)

rows = []
for name, p in MODES:
    X_mode = [to_mode_text(txt, **p) for txt in Xp]

    scores = cross_validate(
        pipe, X_mode, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    rows.append({
        "mode": name,
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "params": p
    })

res = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res[["mode", "accuracy", "macro_f1"]])
display(res[["mode", "params"]])

,mode,accuracy,macro_f1
4,LEMMA_POS_allPOS,0.682540,0.587258
0,BASE_text_allPOS,0.698413,0.554515
3,LEMMA_allPOS,0.666667,0.545224
2,POS_noun_text,0.619048,0.505875
1,POS_content_text,0.603175,0.485663
5,MORPH_NOUN_Case=Nom,0.523810,0.476348
6,MORPH_VERB_Tense=Past,0.238095,0.054945


,mode,params
4,LEMMA_POS_allPOS,"{'allowed_pos': None, 'morph': None, 'repr_mod..."
0,BASE_text_allPOS,"{'allowed_pos': None, 'morph': None, 'repr_mod..."
3,LEMMA_allPOS,"{'allowed_pos': None, 'morph': None, 'repr_mod..."
2,POS_noun_text,"{'allowed_pos': {'NOUN'}, 'morph': None, 'repr..."
1,POS_content_text,"{'allowed_pos': {'ADJ', 'NOUN', 'ADV', 'VERB',..."
5,MORPH_NOUN_Case=Nom,"{'allowed_pos': {'NOUN'}, 'morph': {'Case': {'..."
6,MORPH_VERB_Tense=Past,"{'allowed_pos': {'VERB'}, 'morph': {'Tense': {..."


Наилучшее качество по macro-F1 показал режим LEMMA_POS_allPOS: macro-F1 = 0.587, при accuracy = 0.683.
Это означает, что совмещение леммы и POS-тега повышает устойчивость классификации по всем классам:

## Пункт 4. Нормализация текста

В данном пункте сравниваются разные подходы к нормализации токенов при прочих равных условиях:
- **лемматизация** (приведение к начальной форме; используется `spaCy`);
- **стемминг** (усечение основы слова; используется стеммер из `NLTK`).


In [55]:
import pandas as pd
from nltk.stem.snowball import SnowballStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC




### Подготовка данных для сравнения нормализации

Фиксируются: выбранная предобработка, схема кросс‑валидации и модель. Далее формируются варианты токенизатора, отличающиеся способом нормализации (lemma/stem), чтобы оценить влияние именно этого этапа.

In [113]:
BEST_PREPROCESS = dict(
    lower=True, strip_html=True, remove_urls_emails=True,
    remove_section_numbers=False, normalize_dashes_quotes=True,
    handle_numbers="mask", remove_codelike=False,
    keep_punct=False, normalize_ws=True
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = pd.Series(X).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
cv = StratifiedKFold(n_splits=5 if min_class >= 5 else 3, shuffle=True, random_state=42)



TypeError: preprocess() got an unexpected keyword argument 'lower'

### Стемминг как вариант нормализации

Подключается стеммер (`NLTK`) как альтернативный метод нормализации.

In [57]:
snowball = SnowballStemmer(language="russian")



### Унифицированный токенизатор spaCy с переключаемым режимом

Реализуется функция токенизации, которая в зависимости от режима возвращает либо леммы (`lemma`), либо стеммы (`stem`). Это упрощает сравнение, так как остальная часть конвейера остаётся неизменной.

In [58]:
def make_tokenizer(mode: str):
    def tok(text: str):
        doc = nlp(text)
        tokens = [t for t in doc if not (t.is_space or t.is_punct)]

        if mode == "text":
            return [t.text if t.text != "NUM" else "NUM" for t in tokens]
        if mode == "lemma":
            return [("NUM" if t.text == "NUM" else t.lemma_) for t in tokens]
        return [("NUM" if t.text == "NUM" else snowball.stem(t.text)) for t in tokens]
    return tok


### Режим `stem`

В режиме стемминга токены нормализуются стеммером. Далее они поступают во векторизатор и модель так же, как и при лемматизации, что обеспечивает корректное сравнение подходов.

### Контрольный пример токенизации

Для проверки корректности нормализации выводится пример токенов для одного текста. Это позволяет убедиться, что выбранный режим действительно меняет представление слов (лемма vs стем).

In [59]:
sample = Xp[0]
print("TEXT :", make_tokenizer("text")(sample)[:15])
print("LEMMA:", make_tokenizer("lemma")(sample)[:15])
print("STEM :", make_tokenizer("stem")(sample)[:15])



TEXT : ['полное', 'наименование', 'системы', 'автоматизированное', 'рабочее', 'место', 'продавца', 'консультанта', 'в', 'салоне', 'фотоуслуг', 'в', 'среде', 'NUM', 'с']
LEMMA: ['полный', 'наименование', 'система', 'автоматизированный', 'рабочий', 'место', 'продавец', 'консультант', 'в', 'салон', 'фотоуслуг', 'в', 'среда', 'NUM', 'с']
STEM : ['полн', 'наименован', 'систем', 'автоматизирова', 'рабоч', 'мест', 'продавц', 'консультант', 'в', 'салон', 'фотоуслуг', 'в', 'сред', 'NUM', 'с']


### Оценка качества для вариантов нормализации

Качество сравнивается по кросс‑валидации (`Accuracy`, `macro‑F1`) при фиксированной модели.


In [60]:
rows = []
for mode in ["text", "lemma", "stem"]:
    pipe = make_pipeline(
        TfidfVectorizer(
            sublinear_tf=True,
            tokenizer=make_tokenizer(mode),
            token_pattern=None,
            lowercase=False,
        ),
        LinearSVC(class_weight="balanced"),
    )

    scores = cross_validate(
        pipe, Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )

    rows.append({
        "mode": mode,
        "acc_mean": scores["test_acc"].mean(),
        "acc_std":  scores["test_acc"].std(),
        "f1_mean":  scores["test_f1m"].mean(),
        "f1_std":   scores["test_f1m"].std(),
    })

res = pd.DataFrame(rows).sort_values(["f1_mean", "acc_mean"], ascending=False)
display(res)

best = res.iloc[0]
print(f"Лучший режим: {best['mode']} | macro-F1={best['f1_mean']:.4f}±{best['f1_std']:.4f}, acc={best['acc_mean']:.4f}±{best['acc_std']:.4f}")

,mode,acc_mean,acc_std,f1_mean,f1_std
1,lemma,0.666667,0.077762,0.537319,0.033562
2,stem,0.666667,0.077762,0.537319,0.033562
0,text,0.682540,0.097848,0.527115,0.077173


Лучший режим: lemma | macro-F1=0.5373±0.0336, acc=0.6667±0.0778


### Вывод по пункту 4
лучший метод нормализации (по macro-F1) — лемматизация с добавлением POS (lemma_pos): macro-F1 = 0.587, что выше базового текста на +0.032 и выше простой лемматизации на +0.042.

## Пункт 5. Фильтрация текста

Цель пункта — оценить влияние фильтрации токенов на качество решения и размерность признакового пространства. Рассматриваются типовые приёмы:
- исключение стоп‑слов;
- фильтр по длине токена;
- исключение/сохранение числовых маркеров (например, `NUM`);
- частотные ограничения (через параметры векторизатора `min_df/max_df`, если применимо).

In [61]:
import pandas as pd
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC



### Подготовка окружения spaCy

Используется ранее загруженная модель `spaCy`.

In [62]:
sp = spacy.load("ru_core_news_lg", disable=["parser", "ner"])
print("OK:", sp.pipe_names)



OK: ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']


### Фиксация предобработки и подготовка корпуса

Выбирается одна конфигурация предобработки (лучшая по результатам пункта 2) и формируется массив `Xp`, который далее используется во всех экспериментах по фильтрации без повторного чтения исходного документа.

In [63]:
BEST_PREPROCESS = dict(
    lower=True,
    strip_html=True,
    remove_urls_emails=True,
    remove_section_numbers=False,
    normalize_dashes_quotes=True,
    handle_numbers="mask",     # числа -> NUM
    remove_codelike=False,
    keep_punct=False,          # пунктуацию фиксируем
    normalize_ws=True
)

X = df["text"].astype(str).values
y = df["label"].astype(str).values

Xp = df["text"].apply(lambda t: preprocess(t, **BEST_PREPROCESS)).values

min_class = pd.Series(y).value_counts().min()
n_splits = 5 if min_class >= 5 else 3
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold(n_splits={n_splits}), min_samples_per_class={min_class}")



CV: StratifiedKFold(n_splits=3), min_samples_per_class=3


### Токенизатор с параметрами фильтрации

Определяется функция токенизации (лемматизация фиксирована), внутри которой добавляются фильтры. Это позволяет сравнивать конфигурации фильтрации, не изменяя модель и векторизацию.

In [64]:
def tok(text, use_stop=False, min_len=1, drop_NUM=False, drop_digits=False):
    doc = sp(text)
    out = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        raw = t.text
        if drop_NUM and raw == "NUM":
            continue

        lemma = (t.lemma_ or raw).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if use_stop and t.is_stop:
            continue

        if drop_digits and (t.is_digit or any(ch.isdigit() for ch in lemma)):
            continue

        if len(lemma) < min_len:
            continue

        out.append(lemma)
    return out



### Конфигурации фильтрации

Задаётся несколько конфигураций фильтрации (например: стоп‑слова включены/выключены, минимальная длина токена, обработка `NUM`). Для каждой конфигурации вычисляются метрики качества и оценивается размер словаря.

In [65]:
FILTERS = [
    {"name": "F0_base",        "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F1_stopwords",   "use_stop": True,  "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F2_len>=2",      "use_stop": False, "min_len": 2, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F3_len>=3",      "use_stop": False, "min_len": 3, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F4_drop_NUM",    "use_stop": False, "min_len": 1, "drop_NUM": True,  "drop_digits": False, "min_df": 1, "max_df": 1.0},
    {"name": "F5_min_df=2",    "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 2, "max_df": 1.0},
    {"name": "F6_max_df=0.9",  "use_stop": False, "min_len": 1, "drop_NUM": False, "drop_digits": False, "min_df": 1, "max_df": 0.9},
]



### Сравнение конфигураций фильтрации

Для каждой конфигурации выполняется кросс‑валидационная оценка. В таблицу результатов включаются `Accuracy`, `macro‑F1` и дополнительные показатели, позволяющие сопоставить качество и сложность модели (например, число признаков).

In [68]:
rows = []
for cfg in FILTERS:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        sublinear_tf=True,
        tokenizer=lambda s, c=cfg: tok(  # фиксируем cfg через дефолт-аргумент
            s,
            use_stop=c["use_stop"],
            min_len=c["min_len"],
            drop_NUM=c["drop_NUM"],
            drop_digits=c["drop_digits"],
        ),
        token_pattern=None,
        min_df=cfg["min_df"],
        max_df=cfg["max_df"],
    )

    pipe = make_pipeline(vectorizer, LinearSVC(class_weight="balanced"))

    scores = cross_validate(
        pipe, Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )
    pipe.fit(Xp, y)
    n_features = len(pipe.named_steps["tfidfvectorizer"].get_feature_names_out())

    rows.append({
        "config": cfg["name"],
        "accuracy": scores["test_acc"].mean(),
        "macro_f1": scores["test_f1m"].mean(),
        "n_features": n_features,
        "min_df": cfg["min_df"],
        "max_df": cfg["max_df"],
        "use_stop": cfg["use_stop"],
        "min_len": cfg["min_len"],
        "drop_NUM": cfg["drop_NUM"],
        "drop_digits": cfg["drop_digits"],
    })

res = pd.DataFrame(rows).sort_values(["macro_f1", "accuracy"], ascending=False)
display(res[["config", "accuracy", "macro_f1", "n_features"]])
display(res[["config", "use_stop", "min_len", "drop_NUM", "drop_digits", "min_df", "max_df"]])

best = res.iloc[0]
print(f"BEST: {best['config']} | macro-F1={best['macro_f1']:.4f}, acc={best['accuracy']:.4f}, n_features={int(best['n_features'])}")



,config,accuracy,macro_f1,n_features
1,F1_stopwords,0.730159,0.625259,209
3,F3_len>=3,0.698413,0.607480,230
4,F4_drop_NUM,0.619048,0.550923,253
2,F2_len>=2,0.666667,0.545224,246
0,F0_base,0.666667,0.537319,254
6,F6_max_df=0.9,0.666667,0.537319,254
5,F5_min_df=2,0.555556,0.403666,78


,config,use_stop,min_len,drop_NUM,drop_digits,min_df,max_df
1,F1_stopwords,True,1,False,False,1,1.0
3,F3_len>=3,False,3,False,False,1,1.0
4,F4_drop_NUM,False,1,True,False,1,1.0
2,F2_len>=2,False,2,False,False,1,1.0
0,F0_base,False,1,False,False,1,1.0
6,F6_max_df=0.9,False,1,False,False,1,0.9
5,F5_min_df=2,False,1,False,False,2,1.0


BEST: F1_stopwords | macro-F1=0.6253, acc=0.7302, n_features=209


### Оценка размерности признакового пространства (`n_features`)

Дополнительно фиксируется размер словаря/число признаков после обучения векторизатора на всём корпусе. Это позволяет сопоставить качество и вычислительную сложность выбранного варианта фильтрации.

### Вывод по пункту 5

Проверены различные варианты фильтрации токенов. Сравнение показало, что фильтрация влияет как на качество (метрики), так и на размер словаря (`n_features`). По таблице результатов выбирается конфигурация фильтрации, обеспечивающая наилучший `macro‑F1` при разумной сложности модели.

## Пункт 6. Влияние N‑грамм

На данном шаге исследуется влияние формирования N‑грамм на качество решения. Для чистоты эксперимента изменяется **только** параметр `ngram_range`, а предобработка, токенизация, фильтрация и модель фиксируются.

Сравниваются униграммы, униграммы+биграммы и (при необходимости) триграммы. Помимо метрик качества оценивается рост размерности признакового пространства.

In [69]:
import time
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC




### Подготовка данных и схемы разбиения

Фиксируются корпус `Xp`, целевая переменная `y` и параметры кросс‑валидации. Это обеспечивает корректное сравнение результатов при изменении только `ngram_range`.

In [70]:
Xp = df["text"].astype(str).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values
y  = df["label"].astype(str).values

min_class = pd.Series(y).value_counts().min()
n_splits = 5 if min_class >= 5 else 3
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold, n_splits={n_splits} (min_samples_per_class={min_class})")




CV: StratifiedKFold, n_splits=3 (min_samples_per_class=3)


### Фиксированный токенизатор (лемматизация + стоп‑слова)

Используется один и тот же токенизатор на основе `spaCy` (лемматизация и удаление стоп‑слов). Это позволяет интерпретировать изменение метрик как эффект именно этапа N‑грамм.

In [72]:
STOP_SET = getattr(nlp.Defaults, "stop_words", set())

def tokenizer_lemma(text: str):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        if t.text == "NUM":          # NUM оставляем (как у тебя: DROP_NUM=False)
            tokens.append("NUM")
            continue

        lemma = (t.lemma_ or t.text).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if t.is_stop or lemma in STOP_SET:
            continue

        tokens.append(lemma)
    return tokens




### Эксперименты по `ngram_range`

Перебираются значения `ngram_range` и для каждого варианта оцениваются `Accuracy`, `macro‑F1` и число признаков. По совокупности показателей выбирается наиболее удачный режим.

In [93]:
NGRAMS = [
    ("(1,1)", (1, 1)),
    ("(1,2)", (1, 2)),
    ("(1,3)", (1, 3)),
    ("(2,2)", (2, 2)),  # можно убрать, если не нужно
]

rows = []
for name, rng in NGRAMS:
    tfidf_vectorizer = TfidfVectorizer(
        tokenizer=tokenizer_lemma,
        token_pattern=None,
        lowercase=False,
        sublinear_tf=True,
        ngram_range=rng,
        min_df=1,
        max_df=1.0,
    )
    svc = LinearSVC(class_weight="balanced")
    pipeline = make_pipeline(tfidf_vectorizer, svc)

    t0 = time.perf_counter()
    scores = cross_validate(
        pipeline,
        Xp, y,
        cv=cv,
        scoring={"acc": "accuracy", "f1m": "f1_macro"},
        return_train_score=False
    )
    cv_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    pipeline.fit(Xp, y)
    fit_time = time.perf_counter() - t1

    n_features = len(tfidf_vectorizer.get_feature_names_out())

    rows.append({
        "ngram_range": name,
        "cv_folds": n_splits,
        "accuracy_mean": scores["test_acc"].mean(),
        "accuracy_std":  scores["test_acc"].std(),
        "macro_f1_mean": scores["test_f1m"].mean(),
        "macro_f1_std":  scores["test_f1m"].std(),
        "n_features": n_features,
        "cv_time_sec": cv_time,
        "fit_time_sec": fit_time,
    })

ng_res = pd.DataFrame(rows).sort_values(["macro_f1_mean", "accuracy_mean"], ascending=False)
display(ng_res)

best = ng_res.iloc[0]
print(
    f"ЛУЧШИЙ ngram_range: {best['ngram_range']} | "
    f"macro-F1={best['macro_f1_mean']:.4f}±{best['macro_f1_std']:.4f}, "
    f"acc={best['accuracy_mean']:.4f}±{best['accuracy_std']:.4f}, "
    f"n_features={int(best['n_features'])}, "
    f"cv_time={best['cv_time_sec']:.2f}s"
)

NGRAM_RANGE = (1,2)

,ngram_range,cv_folds,accuracy_mean,accuracy_std,macro_f1_mean,macro_f1_std,n_features,cv_time_sec,fit_time_sec
1,"(1,2)",3,0.730159,0.080937,0.651460,0.085457,494,0.756860,0.223281
0,"(1,1)",3,0.730159,0.059391,0.625259,0.056033,209,0.890621,0.240228
2,"(1,3)",3,0.714286,0.077762,0.601677,0.073197,743,0.648941,0.201067
3,"(2,2)",3,0.380952,0.077762,0.292864,0.060594,285,0.607150,0.205040


ЛУЧШИЙ ngram_range: (1,2) | macro-F1=0.6515±0.0855, acc=0.7302±0.0809, n_features=494, cv_time=0.76s


### Вывод по пункту 6

Показано влияние формирования N‑грамм на качество классификации и размерность признакового пространства. По результатам эксперимента выбирается режим `ngram_range`, обеспечивающий наилучший компромисс между `macro‑F1` и ростом числа признаков; выбранный режим фиксируется далее в пунктах 7–8.

## Пункт 7. Индексирование (векторизация)

В пункте сравниваются методы индексирования текста:
- **BoW (мешок слов)** — `CountVectorizer`;
- **TF‑IDF (частотный портрет)** — `TfidfVectorizer`.

Для сохранения корректности эксперимента фиксируется лучший режим N‑грамм (из пункта 6), а также модель и токенизатор. Дополнительно проверяется влияние параметров векторизации (`min_df`, `max_df`, `max_features`).

In [94]:
import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_validate
from sklearn.svm import LinearSVC




### Подготовка данных

Используются массивы `Xp` и `y`, сформированные ранее. Далее выполняется сравнение векторизаторов на единой схеме кросс‑валидации.

In [95]:
if df is None or len(df) == 0:
    raise ValueError("df пустой: сначала сформируй датасет (text/label)")

X = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = df["text"].apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

TOP_N = 20  # для "портрета"




### Токенизатор для векторизации (леммы + стоп‑слова)

Определяется простой токенизатор `tok`, возвращающий список лемм и исключающий стоп‑слова. Такой формат соответствует требованиям векторизаторов `scikit-learn` и обеспечивает воспроизводимость экспериментов.

In [96]:
def tok(text: str):
    doc = nlp(text)
    tokens = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue
        if t.text == "NUM":
            tokens.append("NUM")
            continue
        if t.is_stop:
            continue

        lemma = (t.lemma_ or "").strip()
        if lemma and lemma != "-PRON-":
            tokens.append(lemma)
    return tokens

bow_vect = CountVectorizer(
    tokenizer=tok,
    token_pattern=None,
    lowercase=False,
    ngram_range=NGRAM_RANGE
)

tfidf_vect = TfidfVectorizer(
    tokenizer=tok,
    token_pattern=None,
    lowercase=False,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=True,
    norm="l2"
)

pipe_bow = make_pipeline(bow_vect, LinearSVC(class_weight="balanced"))
pipe_tfidf = make_pipeline(tfidf_vect, LinearSVC(class_weight="balanced"))

scores_bow = cross_validate(pipe_bow, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})
scores_tfidf = cross_validate(pipe_tfidf, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})


### Обработка числового маркера `NUM`

В технических документах числовые значения встречаются часто. Чтобы сохранить информацию о наличии чисел, используется единый токен `NUM`, который рассматривается как обычный признак.

### Исключение стоп‑слов

Стоп‑слова удаляются на этапе токенизации (`t.is_stop`). Это снижает шум и уменьшает размер словаря, что обычно благоприятно влияет на устойчивость модели.

### Сравнение BoW и TF‑IDF

Для обоих векторизаторов строится конвейер с одинаковым классификатором (`LinearSVC`) и выполняется кросс‑валидационная оценка. Сравниваются `macro‑F1`, `Accuracy` и число признаков.

### Размерность признакового пространства

После обучения векторизатора на полном корпусе фиксируется число признаков (`n_features`). Этот показатель необходим для сопоставления качества и сложности модели.

In [97]:
bow_mat = sparse.csr_matrix(bow_vect.fit_transform(Xp))
tfidf_mat = sparse.csr_matrix(tfidf_vect.fit_transform(Xp))

base_res = pd.DataFrame([
    {
        "vectorizer": "BoW (CountVectorizer)",
        "accuracy": scores_bow["test_acc"].mean(),
        "macro_f1": scores_bow["test_f1m"].mean(),
        "n_features": bow_mat.shape[1]
    },
    {
        "vectorizer": "TF-IDF",
        "accuracy": scores_tfidf["test_acc"].mean(),
        "macro_f1": scores_tfidf["test_f1m"].mean(),
        "n_features": tfidf_mat.shape[1]
    }
]).sort_values(["macro_f1", "accuracy"], ascending=False)

display(base_res)



,vectorizer,accuracy,macro_f1,n_features
1,TF-IDF,0.730159,0.65146,494
0,BoW (CountVectorizer),0.634921,0.48788,494


### Иллюстрация матрицы признаков

Для контроля корректности индексирования выводится небольшой фрагмент матрицы признаков (например, для двух документов).

In [98]:
bow_df = pd.DataFrame(
    bow_mat[:2].toarray(),
    index=["doc1", "doc2"],
    columns=bow_vect.get_feature_names_out()
)
display(bow_df.iloc[:, :20])




,NUM,NUM NUM,NUM hz,NUM me,NUM v,NUM xp,NUM апрель,NUM год,NUM июнь,NUM мб,NUM предприятие,hz,ibm,ibm совместимый,me,me NUM,ms,ms windows,pentium,pentium NUM
doc1,3,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
doc2,4,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### Мини‑сетка гиперпараметров векторизации

Проверяется несколько комбинаций `min_df`, `max_df`, `max_features` отдельно для BoW и TF‑IDF. По результатам выбираются настройки, обеспечивающие наилучший `macro‑F1` при разумной размерности словаря.

In [99]:

GRID = [
    (1, 1.0, None),
    (2, 1.0, None),
    (1, 0.9, None),
    (2, 0.9, None),
    (1, 1.0, 300),
    (1, 1.0, 500),
]

rows = []

for kind in ["BoW", "TF-IDF"]:
    for min_df, max_df, max_features in GRID:

        if kind == "BoW":
            vect = CountVectorizer(
                tokenizer=tok, token_pattern=None, lowercase=False,
                ngram_range=NGRAM_RANGE,
                min_df=min_df, max_df=max_df, max_features=max_features
            )
        else:
            vect = TfidfVectorizer(
                tokenizer=tok, token_pattern=None, lowercase=False,
                ngram_range=NGRAM_RANGE,
                min_df=min_df, max_df=max_df, max_features=max_features,
                sublinear_tf=True, norm="l2"
            )

        pipe = make_pipeline(vect, LinearSVC(class_weight="balanced"))
        scores = cross_validate(pipe, Xp, y, cv=cv, scoring={"acc": "accuracy", "f1m": "f1_macro"})

        mat = sparse.csr_matrix(vect.fit_transform(Xp))
        rows.append({
            "vectorizer": kind,
            "min_df": min_df,
            "max_df": max_df,
            "max_features": max_features,           # тут может быть None
            "accuracy": scores["test_acc"].mean(),
            "macro_f1": scores["test_f1m"].mean(),
            "n_features": mat.shape[1]
        })

grid_res = (pd.DataFrame(rows)
            .sort_values(["vectorizer", "macro_f1", "accuracy"], ascending=[True, False, False]))
display(grid_res)

best_tfidf = grid_res[grid_res["vectorizer"] == "TF-IDF"].iloc[0]
print("Лучший TF-IDF по macro-F1:")
display(pd.DataFrame([best_tfidf]))




,vectorizer,min_df,max_df,max_features,accuracy,macro_f1,n_features
0,BoW,1,1.0,NaN,0.634921,0.487880,494
2,BoW,1,0.9,NaN,0.634921,0.487880,494
5,BoW,1,1.0,500.0,0.634921,0.487880,494
1,BoW,2,1.0,NaN,0.650794,0.470022,71
3,BoW,2,0.9,NaN,0.650794,0.470022,71
4,BoW,1,1.0,300.0,0.619048,0.438940,300
6,TF-IDF,1,1.0,NaN,0.730159,0.651460,494
8,TF-IDF,1,0.9,NaN,0.730159,0.651460,494
11,TF-IDF,1,1.0,500.0,0.730159,0.651460,494
10,TF-IDF,1,1.0,300.0,0.698413,0.560469,300


Лучший TF-IDF по macro-F1:


,vectorizer,min_df,max_df,max_features,accuracy,macro_f1,n_features
6,TF-IDF,1,1.0,NaN,0.730159,0.65146,494


### TF‑IDF «частотный портрет» терминов

Для лучшей конфигурации TF‑IDF строится список наиболее значимых терминов:
- **глобально** (средний TF‑IDF по корпусу),
- **по классам** (средний TF‑IDF внутри каждого раздела).

Этот шаг используется для содержательной интерпретации того, какие слова/словосочетания характеризуют разделы документа.

### Примечание по типам данных (`max_features`)

При сохранении результатов в `DataFrame` значение `None` может преобразоваться в `NaN`. Перед повторным созданием векторизатора параметр `max_features` приводится к корректному виду (`None` или `int`).

In [100]:
mf = best_tfidf["max_features"]
if mf is None or pd.isna(mf):
    mf = None
else:
    mf = int(mf)

best_vect = TfidfVectorizer(
    tokenizer=tok, token_pattern=None, lowercase=False,
    ngram_range=NGRAM_RANGE,
    min_df=int(best_tfidf["min_df"]),
    max_df=float(best_tfidf["max_df"]),
    max_features=mf,                 # <-- исправлено
    sublinear_tf=True, norm="l2"
)

X_tfidf = sparse.csr_matrix(best_vect.fit_transform(Xp))
terms = best_vect.get_feature_names_out()



### Глобальный портрет (топ‑термы по среднему TF‑IDF)

Ранжирование терминов по среднему значению TF‑IDF позволяет выделить наиболее характерные слова/словосочетания корпуса в целом.

In [101]:
global_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
top_idx = global_scores.argsort()[::-1][:TOP_N]
global_portrait = pd.DataFrame({
    "term": terms[top_idx],
    "mean_tfidf": global_scores[top_idx]
})
print("TF-IDF портрет (глобально):")
display(global_portrait)



TF-IDF портрет (глобально):


,term,mean_tfidf
0,NUM,0.059348
1,требование,0.045779
2,справочник,0.043175
3,заказ,0.038066
4,инструкция,0.035352
5,пользователь,0.034244
6,система,0.034205
7,обеспечение,0.025719
8,работа,0.024224
9,NUM NUM,0.022722


### Портрет по классам (разделам)

Для каждого класса вычисляется средний TF‑IDF и формируется список топ‑термов. Полученные списки используются для интерпретации и последующего анализа ошибок классификации.

In [102]:
class_rows = []
for cls in sorted(pd.unique(y)):
    cls_scores = np.asarray(X_tfidf[y == cls].mean(axis=0)).ravel()
    top_idx = cls_scores.argsort()[::-1][:TOP_N]

    for rank, i in enumerate(top_idx, start=1):
        class_rows.append({
            "label": cls,
            "label_title": CHAPTER_MAP.get(cls, ""),
            "rank": rank,
            "term": terms[i],
            "mean_tfidf_in_class": cls_scores[i]
        })

class_portrait = pd.DataFrame(class_rows)
print("TF-IDF портрет (по классам):")
display(class_portrait.head(60))

TF-IDF портрет (по классам):


,label,label_title,rank,term,mean_tfidf_in_class
0,2.2.1,Общие сведения,1,NUM,0.156445
1,2.2.1,Общие сведения,2,работа,0.101579
2,2.2.1,Общие сведения,3,NUM NUM,0.100202
3,2.2.1,Общие сведения,4,заказчик,0.083667
4,2.2.1,Общие сведения,5,наименование,0.071697
5,2.2.1,Общие сведения,6,порядок,0.062930
6,2.2.1,Общие сведения,7,предприятие,0.062677
7,2.2.1,Общие сведения,8,NUM год,0.060913
8,2.2.1,Общие сведения,9,год,0.060913
9,2.2.1,Общие сведения,10,система,0.060103


По пункту 7 (индексирование) лучше TF-IDF, потому что он даёт наибольший macro-F1 (главная метрика при дисбалансе классов) по сравнению с BoW.

Лучший выбор для дальнейшей работы: TF-IDF (TfidfVectorizer, sublinear_tf=True, norm="l2").

Почему: по твоей таблице сравнения/сетки у TF-IDF получился самый высокий macro-F1, значит он лучше различает все разделы ТЗ, включая редкие.

BoW (CountVectorizer) уступает: он чаще даёт ниже macro-F1, потому что учитывает только частоты и хуже “взвешивает” информативные термины.

## Пункт 8. Обучение модели и перебор гиперпараметров

На заключительном этапе выполняется обучение модели с учителем и подбор комбинаций гиперпараметров. Перебираются варианты:
- метод индексирования (TF‑IDF или BoW),
- тип токенизации (леммы / леммы с POS),
- параметры векторизации (`ngram_range`, `min_df`, `max_df`, `max_features`),
- модель и её гиперпараметры (например, `LinearSVC(C)`, `MultinomialNB(alpha)`).

Критерий выбора лучшей конфигурации — максимальный `macro‑F1` по кросс‑валидации. После выбора лучшей конфигурации проводится **OOF‑оценка** (out‑of‑fold): строятся `classification_report` и `confusion_matrix`, что позволяет проанализировать характер ошибок по классам.

In [103]:

import warnings
import numpy as np
import pandas as pd

from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix


warnings.filterwarnings("ignore", category=ConvergenceWarning)

BEST_PREPROCESS = dict(
    lower=True,
    strip_html=True,
    remove_urls_emails=True,
    remove_section_numbers=False,
    normalize_dashes_quotes=True,
    handle_numbers="mask",
    remove_codelike=False,
    keep_punct=False,
    normalize_ws=True
)

X_raw = df["text"].astype(str).values
y = df["label"].astype(str).values
Xp = pd.Series(X_raw).apply(lambda s: preprocess(s, **BEST_PREPROCESS)).values

STOP_SET = set(getattr(nlp.Defaults, "stop_words", set()))

def _tokenize(text: str, repr_mode="lemma", use_stopwords=True, drop_NUM=False, min_len=1):
    doc = nlp(text)
    out = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue

        raw = (t.text or "").strip()
        if not raw:
            continue

        if raw == "NUM":
            if drop_NUM:
                continue
            out.append("NUM")
            continue

        lemma = (t.lemma_ or raw).strip()
        if not lemma or lemma == "-PRON-":
            continue

        if use_stopwords and (t.is_stop or lemma in STOP_SET):
            continue

        if len(lemma) < min_len:
            continue

        if repr_mode == "lemma":
            out.append(lemma)
        elif repr_mode == "lemma_pos":
            out.append(f"{lemma}_{t.pos_}")
        else:
            raise ValueError("repr_mode must be 'lemma' or 'lemma_pos'")
    return out

def tok_lemma_stop(text):     return _tokenize(text, repr_mode="lemma", use_stopwords=True,  drop_NUM=False, min_len=1)
def tok_lemmaPOS_stop(text):  return _tokenize(text, repr_mode="lemma_pos", use_stopwords=True, drop_NUM=False, min_len=1)


min_class = pd.Series(y).value_counts().min()
n_splits = min(5, int(min_class))
if n_splits < 2:
    raise RuntimeError(f"Нельзя делать CV: min_samples_per_class={min_class} < 2")

cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"CV: StratifiedKFold, n_splits={n_splits} (min_samples_per_class={min_class})")

pipe = Pipeline([
    ("vect", TfidfVectorizer(
        lowercase=False,
        tokenizer=tok_lemma_stop,
        token_pattern=None,
        preprocessor=None,
        sublinear_tf=True,
        norm="l2"
    )),
    ("clf", LinearSVC(class_weight="balanced"))
])


def make_logreg():
    return LogisticRegression(
        class_weight="balanced",
        solver="saga",
        max_iter=20000,
    )



param_grid = [
    {
        "vect__tokenizer": [tok_lemma_stop, tok_lemmaPOS_stop],
        "vect__ngram_range": [(1,1), (1,2), (1,3)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500, 300],
        "clf": [LinearSVC(class_weight="balanced")],
        "clf__C": [0.5, 1.0, 2.0, 5.0],
    },

    {
        "vect__tokenizer": [tok_lemma_stop],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [make_logreg()],
        "clf__C": [0.5, 1.0, 2.0, 5.0],
    },

    {
        "vect__tokenizer": [tok_lemma_stop],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [MultinomialNB()],
        "clf__alpha": [0.1, 0.5, 1.0],
    },

    {
        "vect": [CountVectorizer(
            lowercase=False,
            tokenizer=tok_lemma_stop,
            token_pattern=None,
            preprocessor=None
        )],
        "vect__ngram_range": [(1,1), (1,2)],
        "vect__min_df": [1, 2],
        "vect__max_df": [1.0, 0.9],
        "vect__max_features": [None, 500],
        "clf": [MultinomialNB()],
        "clf__alpha": [0.1, 0.5, 1.0],
    },
]

search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    refit=True,
    n_jobs=None,
    verbose=0,
    error_score=np.nan,
    return_train_score=False
)

search.fit(Xp, y)

print("Best CV macro-F1:", search.best_score_)
print("Best params (short):")
for k, v in search.best_params_.items():
    if k in ("vect", "clf"):
        print(f"  {k}: {type(v).__name__}")
    elif k == "vect__tokenizer":
        print(f"  {k}: {'tok_lemmaPOS_stop' if v==tok_lemmaPOS_stop else 'tok_lemma_stop'}")
    else:
        print(f"  {k}: {v}")

best_model = search.best_estimator_

vect = best_model.named_steps["vect"]
try:
    n_features = len(vect.get_feature_names_out())
except Exception:
    n_features = len(getattr(vect, "vocabulary_", {}))
print("Best model n_features:", n_features)

y_pred_oof = cross_val_predict(best_model, Xp, y, cv=cv)

print("\n=== OOF (CV) classification_report ===")
print(classification_report(y, y_pred_oof, digits=4, zero_division=0))

labels_sorted = sorted(pd.unique(y))
cm = confusion_matrix(y, y_pred_oof, labels=labels_sorted)
cm_df = pd.DataFrame(cm, index=[f"true:{l}" for l in labels_sorted], columns=[f"pred:{l}" for l in labels_sorted])

print("\n=== OOF (CV) confusion matrix ===")
display(cm_df)

print("\nLegend (label -> title):")
for l in labels_sorted:
    print(f"{l} -> {CHAPTER_MAP.get(l, '')}")

cv_res = pd.DataFrame(search.cv_results_)
top = cv_res.sort_values("mean_test_score", ascending=False).head(10)[
    ["mean_test_score", "std_test_score", "rank_test_score", "params"]
].reset_index(drop=True)

print("\n=== TOP-10 configs by CV macro-F1 ===")
display(top)

CV: StratifiedKFold, n_splits=3 (min_samples_per_class=3)
Best CV macro-F1: 0.6717858860716004
Best params (short):
  clf: MultinomialNB
  clf__alpha: 0.1
  vect: CountVectorizer
  vect__max_df: 1.0
  vect__max_features: None
  vect__min_df: 1
  vect__ngram_range: (1, 2)
Best model n_features: 494

=== OOF (CV) classification_report ===
              precision    recall  f1-score   support

       2.2.1     1.0000    0.8333    0.9091         6
     2.2.3.1     0.8000    0.3636    0.5000        11
     2.2.3.2     0.9333    0.9333    0.9333        15
     2.2.3.3     0.5217    0.8000    0.6316        15
       2.2.4     1.0000    1.0000    1.0000         3
       2.2.6     0.8889    0.8000    0.8421        10
       2.2.7     0.0000    0.0000    0.0000         3

    accuracy                         0.7302        63
   macro avg     0.7349    0.6758    0.6880        63
weighted avg     0.7701    0.7302    0.7278        63


=== OOF (CV) confusion matrix ===


,pred:2.2.1,pred:2.2.3.1,pred:2.2.3.2,pred:2.2.3.3,pred:2.2.4,pred:2.2.6,pred:2.2.7
true:2.2.1,5,0,0,1,0,0,0
true:2.2.3.1,0,4,0,7,0,0,0
true:2.2.3.2,0,0,14,0,0,1,0
true:2.2.3.3,0,0,0,12,0,0,3
true:2.2.4,0,0,0,0,3,0,0
true:2.2.6,0,1,1,0,0,8,0
true:2.2.7,0,0,0,3,0,0,0



Legend (label -> title):
2.2.1 -> Общие сведения
2.2.3.1 -> Требования к системе
2.2.3.2 -> Требования к функциям
2.2.3.3 -> Требования к обеспечению
2.2.4 -> Этапы работ
2.2.6 -> Подготовка к вводу
2.2.7 -> Документирование

=== TOP-10 configs by CV macro-F1 ===


,mean_test_score,std_test_score,rank_test_score,params
0,0.671786,0.065264,1,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
1,0.671786,0.065264,1,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
2,0.671786,0.065264,1,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
3,0.671786,0.065264,1,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
4,0.666252,0.064644,5,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
5,0.666252,0.064644,5,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
6,0.666252,0.064644,5,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
7,0.666252,0.064644,5,"{'clf': LinearSVC(class_weight='balanced'), 'c..."
8,0.663469,0.069926,9,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."
9,0.663469,0.069926,9,"{'clf': MultinomialNB(), 'clf__alpha': 0.1, 'v..."


## Заключение по работе

В ходе лабораторной работы построен полный конвейер анализа текстов для задачи многоклассовой классификации абзацев технического задания: от формирования корпуса из файла до подбора гиперпараметров и итоговой OOF‑оценки.

**Основные результаты:**
- проведены экспериментальные сравнения на этапах предобработки, POS/морфологии, нормализации, фильтрации, N‑грамм и индексирования;
- подобрана лучшая комбинация параметров по критерию `macro‑F1` (см. TOP‑таблицу);
- выполнен анализ ошибок по матрице несоответствий (confusion matrix), что позволяет выявить разделы, которые модель путает чаще всего.

**Вывод:** качество модели существенно зависит от этапов подготовки данных; наиболее устойчивый результат достигается при согласованном выборе нормализации, фильтрации и параметров векторизации.